<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的配套代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库： <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第 7 章：练习解答

本 notebook 使用的软件包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 供 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
## 练习 7.1：测试 `<think>` 格式奖励

- 以下代码检查：如果 think 词元使用不正确，格式奖励应为零。

In [ ]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(
    kind="reasoning", tokenizer_only=True, out_dir="qwen3"
)

tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

In [ ]:
import torch
from reasoning_from_scratch.ch07 import reward_format

prompt = "Calculate ..."

def check_case(name, rollout):
    token_ids = tokenizer.encode(prompt + rollout)
    prompt_len = len(tokenizer.encode(prompt))

    reward = reward_format(
        token_ids=torch.tensor(token_ids),
        prompt_len=prompt_len,
    )

    print(f"{name}: {reward}")


# 1）正确情况
check_case(
    "Correct order",
    "Let's ... <think> ... </think> ..."
)

# 2）标签拼写错误
check_case(
    "Typo in <think>",
    "Let's ... <thnik> ... </think> ..."
)

# 3）顺序颠倒
check_case(
    "Reversed order",
    "Let's ... </think> ... <think> ..."
)

# 4）缺少一个标签
check_case(
    "Missing </think>",
    "Let's ... <think> ..."
)

- 输出表明，只有正确使用 `<think>...</think>` 标签，函数才会给予 1.0 的奖励。

&nbsp;
## 练习 7.2：条件格式奖励

- 条件奖励的实现非常简单；主体章节讨论的总奖励实现如下：

```python
reward = rlvr_reward + format_reward_weight * format_reward
```

- 因此，如果正确性奖励（`rlvr_reward`）为 0.0，可以用以下方式禁用格式奖励：

```python
if conditional_reward:
    format_reward *= rlvr_reward
reward = rlvr_reward + format_reward_weight * format_reward
```


- 实际使用时，可以启用 `--conditional_reward` 标志运行 [03_rlvr_grpo_scripts_advanced/7_6_plus_format_reward.py](../03_rlvr_grpo_scripts_advanced/7_6_plus_format_reward.py) 脚本（与第 7 章的第 7.6 节类似）
- 为分析结果，先再次绘制第 7.6 章中不使用条件奖励的原始结果作为参考：


In [ ]:
from reasoning_from_scratch.ch07 import download_from_github

download_from_github(
    "ch07/02_logs/7_6_plus_format_reward_metrics.csv"
)

In [ ]:
from reasoning_from_scratch.ch07 import plot_grpo_metrics

plot_grpo_metrics(
    "7_6_plus_format_reward_metrics.csv",
    columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
)

In [ ]:
plot_grpo_metrics(
    "7_6_plus_format_reward_metrics.csv",
    columns=["reward_avg", "format_reward_avg", "adv_std", "entropy_avg"],
)

- 接下来查看启用条件奖励后的训练过程：

In [ ]:
download_from_github(
    "ch07/02_logs/7_6_plus_format_reward_conditional_metrics.csv"
)

In [ ]:
plot_grpo_metrics(
    "7_6_plus_format_reward_conditional_metrics.csv",
    columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
    #save_as="ex07-1.pdf"
)

- 评估准确率和平均奖励大幅下降，但似乎有所恢复
- 总体而言，尽管出现性能骤降，该过程看起来比之前更稳定；趋势表明，如果训练更久，性能还会继续改善


In [ ]:
plot_grpo_metrics(
    "7_6_plus_format_reward_conditional_metrics.csv",
    columns=["reward_avg", "format_reward_avg", "adv_std", "entropy_avg"],
    # save_as="ex07-2.pdf"
)

- 可以看到，平均格式奖励几乎完全复现了平均奖励曲线；这是条件逻辑正常工作的良好健全性检查
- 平均格式奖励还显示了在正确答案子集中，总奖励有多少来自格式项
- 不过，由于平均格式奖励曲线与平均奖励曲线一致，它主要只是一个奖励加成（而且模型正确时似乎总能获得该奖励；这是合理的，因为训练后的推理模型已知道如何正确使用 `<think>...</think>` 标签，而且没有遗忘这一能力）
- 但熵的增加仍然值得警惕，可能暗示训练不稳定；也许可以通过更严格的裁剪（例如更小的 `clip_eps`）等方式解决
